# Calibration of Polymarket sports markets
### Advanced Applied Statistics · research notebook

**Research question.** How well do closed sports markets' probabilities correspond to settled outcomes, and does the relationship differ across sports and final-volume groups?

**Unit of analysis.** For each settled binary market, use every available hourly VWAP in hours **2–24 before the market's `endDate`**. A market contributes at most once to a given 10-percentage-point probability bin; if several of its hours fall in the same bin, their probabilities are averaged. The same market can contribute to different bins. Hour 1 is excluded.

This is a **retrospective, trajectory-selected analysis**. The market's visit to a bin is determined over the entire selected window; it is not a forecast made at a common instant or a trading strategy. Sport labels use Pietro's event-slug-first rules; they are inferred, not verified ground truth.

The notebook uses only saved files. The first full run reads the large hourly master file; it stores a fingerprinted cache for later runs. There is no sample-size cap and no API download.

## 1 · Reproducible setup

Run the notebook from the PolyCats folder, or set `POLYCATS_ROOT` to that folder. Set `POLYCATS_BOOTSTRAPS` before launching Jupyter to change the number of event bootstrap replicates. The default seed and number of replicates are recorded below.

In [ ]:
import os, json, hashlib, importlib.util, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.stats import false_discovery_control
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display

ROOT = Path(os.environ.get('POLYCATS_ROOT', Path.cwd())).expanduser().resolve()
DATA = ROOT / 'Data' / 'all_closed_sports'
MASTER = DATA / 'closed_sports_all_data.csv'
LINKS = DATA / 'market_events.jsonl'
EVENTS = DATA / 'events.jsonl'
PIPELINE = ROOT / 'Polymarket_Closed_Sports_Combined_Pipeline.ipynb'
PLOT_CODE = ROOT / 'plot_event_sport_calibration.py'
N_BINS, FIRST_HOUR, LAST_HOUR = 10, 2, 24
SEED = 2026
N_BOOT = int(os.environ.get('POLYCATS_BOOTSTRAPS', '500'))
EPS = 1e-6

for path in [MASTER, LINKS, EVENTS, PIPELINE, PLOT_CODE]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Open this notebook from the PolyCats folder or set POLYCATS_ROOT.')
if N_BOOT < 100:
    raise ValueError('Use at least 100 bootstrap replicates; 500 or more is preferable.')

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 200, 'font.family': 'serif',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': .18, 'axes.titlesize': 11,
    'axes.labelsize': 10, 'legend.frameon': False,
})
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print(f'Project: {ROOT}\nHours: {FIRST_HOUR}–{LAST_HOUR} | Probability bins: {N_BINS} | Bootstrap: {N_BOOT} | Seed: {SEED}')
print(f'pandas {pd.__version__} | scipy {scipy.__version__}')

## 2 · Construct the analysis data

For market $i$, hour $h$, and binary first-outcome result $y_i$, let $p_{ih}$ be its saved hourly VWAP. If only the opposite outcome traded, use $1-p_{ih}^{(\mathrm{opposite})}$. Settled outcomes follow the existing pipeline's 0.02/0.98 thresholds.

For each market and bin $b$, the representative price is the mean of its eligible hourly probabilities in that bin, $\bar p_{ib}$. We join the market to its **first saved event ID** and use the existing event-sport labeller. Missing event links or end dates are reported and excluded from analyses that require them. An event ID is a proxy for a game/cluster; its quality should be audited.

In [ ]:
spec = importlib.util.spec_from_file_location('polycats_calibration', PLOT_CODE)
calibration = importlib.util.module_from_spec(spec)
spec.loader.exec_module(calibration)

def source_fingerprint(paths):
    return {str(p): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns} for p in paths}

source_files = [MASTER, LINKS, EVENTS, PIPELINE, PLOT_CODE]
fingerprint = source_fingerprint(source_files)
cache_key = hashlib.sha256(json.dumps({'files': fingerprint, 'bins': N_BINS, 'first_hour': FIRST_HOUR}, sort_keys=True).encode()).hexdigest()[:16]
CACHE = DATA / 'analysis_cache' / f'course_hourly_{cache_key}.pkl'

if CACHE.exists():
    df = pd.read_pickle(CACHE)
    print(f'Loaded local analysis cache: {CACHE}')
else:
    df = calibration.hourly_market_bins(MASTER, n_bins=N_BINS, start_hour=FIRST_HOUR)
    if df.duplicated(['market_id', 'probability_bin']).any():
        raise ValueError('Duplicate market / probability-bin pairs in the master data.')
    wanted = set(df['market_id'].astype(str))
    market_event = {}
    with LINKS.open(encoding='utf-8') as source:
        for line in source:
            if not line.strip():
                continue
            record = json.loads(line)
            mid = str(record.get('id', ''))
            ids = record.get('event_ids') or []
            if mid in wanted and ids:
                market_event.setdefault(mid, str(ids[0]))
    sport_map, _, _ = calibration.event_sports_for_forecasts(df, DATA, PIPELINE)
    df['event_id'] = df['market_id'].astype(str).map(market_event)
    df['sport'] = df['market_id'].astype(str).map(sport_map).fillna('Other / unclassified')

    # The timestamp comes from the same hourly master file as the forecast.
    date_parts = []
    for chunk in pd.read_csv(MASTER, usecols=['id', 'endDate'], dtype={'id': 'string'}, chunksize=100_000, low_memory=False):
        part = chunk.loc[chunk['id'].isin(wanted)].copy()
        if not part.empty:
            date_parts.append(part)
    dates = pd.concat(date_parts, ignore_index=True).rename(columns={'id': 'market_id'})
    if dates['market_id'].duplicated().any():
        raise ValueError('Duplicate market IDs in the hourly master; inspect before joining dates.')
    dates['end_date'] = pd.to_datetime(dates.pop('endDate'), errors='coerce', utc=True)
    df = df.merge(dates, on='market_id', how='left', validate='many_to_one')
    df['volumeNum'] = pd.to_numeric(df['volumeNum'], errors='coerce')
    if source_fingerprint(source_files) != fingerprint:
        raise RuntimeError('Source files changed during processing; rerun for a consistent snapshot.')
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_pickle(CACHE)
    print(f'Saved local analysis cache: {CACHE}')

assert not df.duplicated(['market_id', 'probability_bin']).any()
assert df['probability'].between(0, 1).all() and df['actual'].isin([0, 1]).all()
coverage = pd.Series({
    'Market–bin rows': len(df),
    'Unique markets': df['market_id'].nunique(),
    'Linked events': df['event_id'].nunique(),
    'Markets without an event link': df.loc[df['event_id'].isna(), 'market_id'].nunique(),
    'Markets without an end date': df.loc[df['end_date'].isna(), 'market_id'].nunique(),
    'Markets with unclassified sport': df.loc[df['sport'].eq('Other / unclassified'), 'market_id'].nunique(),
})
display(coverage.to_frame('count'))

**Interpretation and exclusions.** Hourly VWAP is a trade-price proxy, not necessarily the order-book midpoint or a quote available at a fixed moment. The window is relative to `endDate`, not necessarily kick-off. Some trade histories may be truncated at the API limit. The final market metadata reports volume, liquidity and spread at close; it does not reconstruct their historical hourly values. In particular, final volume below is used only for **retrospective stratification**, never as a prospective predictor.

In [ ]:
eligible = df.loc[df['event_id'].notna() & df['end_date'].notna() & df['sport'].ne('Other / unclassified')].copy()
sport_counts = eligible.groupby('sport')['market_id'].nunique().sort_values(ascending=False)
TOP_SPORTS = sport_counts.head(7).index.tolist()
analysis = eligible.loc[eligible['sport'].isin(TOP_SPORTS)].copy()
analysis['residual'] = analysis['actual'] - analysis['probability']
analysis['brier'] = (analysis['probability'] - analysis['actual']) ** 2
print('Seven largest sports by unique markets:', ', '.join(TOP_SPORTS))
display(analysis.groupby('sport').agg(markets=('market_id', 'nunique'), events=('event_id', 'nunique'), market_bin_rows=('market_id', 'size')).sort_values('markets', ascending=False))

## 3 · Calibration and event-cluster bootstrap

In probability bin $b$, estimated calibration is

$$
\widehat r_b=\frac{1}{n_b}\sum_{i\in b} y_i,
\qquad
\widehat p_b=\frac{1}{n_b}\sum_{i\in b}\bar p_{ib}.
$$

The diagonal $\widehat r_b=\widehat p_b$ represents perfect calibration. We resample **event IDs**, retaining all of an event's market–bin rows, then recompute the rate in each bin. The 2.5th and 97.5th percentiles give approximate 95% bootstrap intervals. This accounts for dependence within the saved event ID, but not necessarily dependence between different related event IDs.

In [ ]:
# Assign retrospective volume tertiles once per market, separately within sport.
market_volume = analysis.drop_duplicates('market_id')[['market_id', 'sport', 'volumeNum']].copy()
market_volume['volume_tier'] = 'Missing'
for sport in TOP_SPORTS:
    mask = market_volume['sport'].eq(sport) & market_volume['volumeNum'].ge(0)
    if mask.sum() >= 3:
        market_volume.loc[mask, 'volume_tier'] = pd.qcut(
            market_volume.loc[mask, 'volumeNum'].rank(method='first'), 3,
            labels=['Low', 'Middle', 'High']
        ).astype(str).to_numpy()
analysis['volume_tier'] = analysis['market_id'].map(market_volume.set_index('market_id')['volume_tier'])

clusters, event_ids = pd.factorize(analysis['event_id'], sort=True)
sport_codes = pd.Categorical(analysis['sport'], categories=TOP_SPORTS).codes
bin_codes = analysis['probability_bin'].to_numpy(dtype=int)
y = analysis['actual'].to_numpy(dtype=float)
p = analysis['probability'].to_numpy(dtype=float)
n_events = len(event_ids)
n_sports = len(TOP_SPORTS)

def event_matrix(values, row_index, column_index, n_columns):
    return sparse.coo_matrix((values, (row_index, column_index)), shape=(n_events, n_columns)).tocsr()

cell_codes = sport_codes * N_BINS + bin_codes
counts = event_matrix(np.ones(len(analysis)), clusters, cell_codes, n_sports * N_BINS)
successes = event_matrix(y, clusters, cell_codes, n_sports * N_BINS)
prob_sums = event_matrix(p, clusters, cell_codes, n_sports * N_BINS)

# The test family contains seven all-volume groups and their available volume tertiles.
tier_codes = analysis['volume_tier'].map({'Low': 1, 'Middle': 2, 'High': 3}).fillna(0).to_numpy(dtype=int)
volume_ok = tier_codes > 0
test_rows = np.concatenate([clusters, clusters[volume_ok]])
test_cols = np.concatenate([sport_codes * 4, sport_codes[volume_ok] * 4 + tier_codes[volume_ok]])
test_residuals = np.concatenate([y - p, (y - p)[volume_ok]])
test_counts = event_matrix(np.ones(len(test_rows)), test_rows, test_cols, n_sports * 4)
test_sums = event_matrix(test_residuals, test_rows, test_cols, n_sports * 4)

observed_counts = np.asarray(counts.sum(axis=0)).ravel()
observed_y = np.asarray(successes.sum(axis=0)).ravel()
observed_p = np.asarray(prob_sums.sum(axis=0)).ravel()
test_n = np.asarray(test_counts.sum(axis=0)).ravel()
test_sum = np.asarray(test_sums.sum(axis=0)).ravel()
observed_gap = np.divide(test_sum, test_n, out=np.full_like(test_sum, np.nan), where=test_n > 0)
bootstrap_rates = np.full((N_BOOT, n_sports * N_BINS), np.nan)
bootstrap_gaps = np.full((N_BOOT, n_sports * 4), np.nan)
rng = np.random.default_rng(SEED)
for b in range(N_BOOT):
    weights = np.bincount(rng.integers(0, n_events, size=n_events), minlength=n_events)
    n_star = np.asarray(counts.T @ weights).ravel()
    y_star = np.asarray(successes.T @ weights).ravel()
    bootstrap_rates[b] = np.divide(y_star, n_star, out=np.full_like(y_star, np.nan), where=n_star > 0)
    n_test_star = np.asarray(test_counts.T @ weights).ravel()
    r_test_star = np.asarray(test_sums.T @ weights).ravel()
    bootstrap_gaps[b] = np.divide(r_test_star, n_test_star, out=np.full_like(r_test_star, np.nan), where=n_test_star > 0)
    if (b + 1) % 100 == 0:
        print(f'Bootstrap {b + 1:,}/{N_BOOT:,}', flush=True)
print(f'Resampled {n_events:,} saved event IDs per replicate.')

In [ ]:
rows = []
for j, sport in enumerate(TOP_SPORTS):
    for b in range(N_BINS):
        k = j * N_BINS + b
        if observed_counts[k] == 0:
            continue
        draws = bootstrap_rates[:, k]
        draws = draws[np.isfinite(draws)]
        low, high = np.quantile(draws, [.025, .975]) if len(draws) else (np.nan, np.nan)
        rows.append({'sport': sport, 'bin': b, 'n': int(observed_counts[k]),
                     'mean_probability': observed_p[k] / observed_counts[k],
                     'observed_rate': observed_y[k] / observed_counts[k], 'ci_low': low, 'ci_high': high})
calibration_table = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 4, figsize=(13, 7), sharex=True, sharey=True, constrained_layout=True)
for ax, sport in zip(axes.flat, TOP_SPORTS):
    group = calibration_table.loc[calibration_table['sport'].eq(sport)]
    ax.plot([0, 1], [0, 1], '--', color='0.5', lw=1, label='Perfect calibration')
    ax.vlines(group['mean_probability'], group['ci_low'], group['ci_high'], color='#386a9e', lw=1.1)
    ax.scatter(group['mean_probability'], group['observed_rate'], color='#386a9e', s=16, zorder=3)
    n_markets = analysis.loc[analysis['sport'].eq(sport), 'market_id'].nunique()
    ax.set(title=f'{sport} (markets = {n_markets:,})', xlim=(0, 1), ylim=(0, 1), aspect='equal')
for ax in axes[-1, :]:
    ax.set_xlabel('Predicted probability')
for ax in axes[:, 0]:
    ax.set_ylabel('Observed event frequency')
axes.flat[-1].axis('off')
fig.suptitle('Calibration by event-inferred sport · hours 2–24', fontsize=13)
plt.show()
display(calibration_table.head(10))

## 4 · Final-volume groups: a descriptive comparison

Within each sport, markets are ranked by their **final** reported volume and divided into three equally sized groups. Ties can be split by rank. Each market stays in the same tier across its probability bins. A market with missing volume remains in the all-volume analysis but is excluded from these tertiles.

The Brier score, $n^{-1}\sum (p-y)^2$, describes probabilistic error. It is not a pure calibration metric and is affected by the probability distribution and outcome prevalence. Do not interpret volume comparisons as causal or as real-time predictors: the volume is a closing snapshot.

In [ ]:
volume_summary = (analysis.loc[analysis['volume_tier'].ne('Missing')]
    .groupby(['sport', 'volume_tier'], observed=True)
    .agg(markets=('market_id', 'nunique'), events=('event_id', 'nunique'),
         market_bin_rows=('market_id', 'size'), mean_probability=('probability', 'mean'),
         observed_rate=('actual', 'mean'), calibration_gap=('residual', 'mean'), brier=('brier', 'mean'))
    .reset_index())
display(volume_summary.sort_values(['sport', 'volume_tier']).reset_index(drop=True))

## 5 · Multiple testing: calibration-in-the-large

For each sport and sport × volume group, test the descriptive null $H_0: E[Y-p]=0$. This tests **average calibration error only**; a zero average can hide deviations within individual probability bins, and it is not a direct test that two sports differ.

The approximate two-sided bootstrap p-value compares the observed gap to the centred distribution of event-bootstrap gaps. We correct the resulting family of up to 28 tests using Benjamini–Hochberg (BH) and, more conservatively, Benjamini–Yekutieli (BY). Groups overlap and tests are dependent, so the assumptions behind BH should be considered. These exploratory tests are not a substitute for a pre-registered hypothesis family.

In [ ]:
tests = []
for j, sport in enumerate(TOP_SPORTS):
    for t, tier in enumerate(['All volume', 'Low', 'Middle', 'High']):
        k = j * 4 + t
        if test_n[k] == 0:
            continue
        draws = bootstrap_gaps[:, k]
        draws = draws[np.isfinite(draws)]
        if len(draws) < N_BOOT * .9:
            continue
        gap = observed_gap[k]
        low, high = np.quantile(draws, [.025, .975])
        p_value = (1 + np.count_nonzero(np.abs(draws - gap) >= abs(gap))) / (len(draws) + 1)
        tests.append({'sport': sport, 'volume_tier': tier, 'market_bin_rows': int(test_n[k]),
                      'mean_y_minus_p': gap, 'ci_low': low, 'ci_high': high, 'p_boot': p_value})
tests = pd.DataFrame(tests)
if not tests.empty:
    tests['p_BH'] = false_discovery_control(tests['p_boot'].to_numpy(), method='bh')
    tests['p_BY'] = false_discovery_control(tests['p_boot'].to_numpy(), method='by')
    display(tests.sort_values('p_BH').reset_index(drop=True))
else:
    print('No test group had enough valid event-bootstrap replicates.')

## 6 · Logistic calibration and chronological evaluation

For binary outcome $Y$, logistic calibration models

$$
\operatorname{logit}\Pr(Y=1\mid p)=\alpha+\beta\,\operatorname{logit}(p).
$$

Perfect calibration corresponds to $\alpha=0$, $\beta=1$. The model below is fitted by maximum likelihood. Probabilities exactly 0 or 1 are clipped to $[10^{-6},1-10^{-6}]$ for the logit and log loss; this choice should be checked in sensitivity analysis.

We split **event groups chronologically** using the minimum and maximum market `endDate` in each event: training events must finish before the cutoff, test events must start at least 24 hours after it, and events crossing the boundary or embargo are excluded. The 24-hour embargo separates training end dates from the earliest test-hour window. This protects against event overlap across the model-fitting split, but final outcomes may be resolved after `endDate`; their availability at the training cutoff is not verified. It **does not establish pre-game forecasting** or remove information that may already have entered the original prices.

In [ ]:
spans = (eligible.groupby('event_id')['end_date'].agg(event_min='min', event_max='max').dropna())
cutoff = spans['event_max'].quantile(.80)
train_events = spans.index[spans['event_max'] < cutoff]
test_events = spans.index[spans['event_min'] >= cutoff + pd.Timedelta(hours=24)]
crossing_events = spans.index.difference(train_events.union(test_events))
training_candidates = eligible.loc[eligible['event_id'].isin(train_events)]
model_sports = training_candidates.groupby('sport')['market_id'].nunique().nlargest(7).index.tolist()
model_data = eligible.loc[eligible['sport'].isin(model_sports) & eligible['event_id'].isin(train_events.union(test_events))].copy()
train_mask = model_data['event_id'].isin(train_events).to_numpy()
test_mask = model_data['event_id'].isin(test_events).to_numpy()
assert set(model_data.loc[train_mask, 'event_id']).isdisjoint(model_data.loc[test_mask, 'event_id'])
assert model_data.loc[train_mask, 'end_date'].max() < model_data.loc[test_mask, 'end_date'].min() - pd.Timedelta(hours=24)
print(f'Cutoff: {cutoff}; excluded boundary-spanning events: {len(crossing_events):,}')
print(f'Training rows: {train_mask.sum():,}; test rows: {test_mask.sum():,}')
print('Model sports selected from training events:', ', '.join(model_sports))

model_p = model_data['probability'].to_numpy(dtype=float)
model_y = model_data['actual'].to_numpy(dtype=int)
clipped_p = np.clip(model_p, EPS, 1 - EPS)
z = np.log(clipped_p / (1 - clipped_p))
X_base = z.reshape(-1, 1)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    mle = LogisticRegression(penalty=None, solver='lbfgs', max_iter=500).fit(X_base[train_mask], model_y[train_mask])
print(f'Training logistic calibration: intercept = {mle.intercept_[0]:.4f}; slope = {mle.coef_[0, 0]:.4f}')

## 7 · Shrinkage and event-aware cross-validation

The sport-aware model adds sport indicators and their interactions with $\operatorname{logit}(p)$. Ridge regularisation shrinks coefficients toward zero:

$$
\widehat{\theta}_{\lambda}
=\arg\min_{\theta}\{-\ell(\theta)+\lambda\lVert\theta_{\text{non-intercept}}\rVert_2^2\}.
$$

Here scikit-learn's `C` is an inverse regularisation-strength parameter. We standardise features **inside each training fold** and choose `C` using two expanding chronological folds of the training events. The final held-out period is not used for tuning. The analysis remains retrospective because the market–bin representation uses the whole selected 23-hour window and final `endDate` may follow game start.

In [ ]:
sport_category = pd.Categorical(model_data['sport'], categories=model_sports)
dummies = pd.get_dummies(sport_category, drop_first=True, dtype=float).to_numpy()
X_sport = np.column_stack([z, dummies, z[:, None] * dummies])
train_span = spans.loc[train_events]
fold_cuts = [train_span['event_max'].quantile(.50), train_span['event_max'].quantile(.75), cutoff]
folds = []
for lo, hi in zip(fold_cuts[:-1], fold_cuts[1:]):
    fit_events = train_span.index[train_span['event_max'] < lo]
    val_events = train_span.index[(train_span['event_min'] >= lo + pd.Timedelta(hours=24)) & (train_span['event_max'] < hi)]
    fit_mask = model_data['event_id'].isin(fit_events).to_numpy()
    val_mask = model_data['event_id'].isin(val_events).to_numpy()
    if not fit_mask.any() or not val_mask.any() or len(np.unique(model_y[fit_mask])) < 2:
        raise ValueError('A chronological fold has insufficient events or outcomes; adjust fold cutoffs.')
    assert model_data.loc[fit_mask, 'end_date'].max() < model_data.loc[val_mask, 'end_date'].min() - pd.Timedelta(hours=24)
    folds.append((fit_mask, val_mask))
    print(f'Fold: train={fit_mask.sum():,} rows; validate={val_mask.sum():,} rows; cutoff={lo}')

candidates = [0.01, 0.1, 1.0]
cv_rows = []
for C in candidates:
    for fold, (fit_mask, val_mask) in enumerate(folds, 1):
        ridge = make_pipeline(StandardScaler(), LogisticRegression(C=C, solver='lbfgs', max_iter=500))
        ridge.fit(X_sport[fit_mask], model_y[fit_mask])
        predictions = ridge.predict_proba(X_sport[val_mask])[:, 1]
        cv_rows.append({'C': C, 'fold': fold, 'validation_rows': int(val_mask.sum()),
                        'log_loss': log_loss(model_y[val_mask], predictions, labels=[0, 1])})
cv_table = pd.DataFrame(cv_rows)
cv_scores = (cv_table.assign(weighted_loss=cv_table['log_loss'] * cv_table['validation_rows'])
             .groupby('C')[['weighted_loss', 'validation_rows']].sum())
cv_scores['mean_log_loss'] = cv_scores['weighted_loss'] / cv_scores['validation_rows']
best_C = float(cv_scores['mean_log_loss'].idxmin())
display(cv_table)
display(cv_scores)
print(f'Selected C: {best_C}')

ridge_final = make_pipeline(StandardScaler(), LogisticRegression(C=best_C, solver='lbfgs', max_iter=500))
ridge_final.fit(X_sport[train_mask], model_y[train_mask])
predictions = {
    'Original market probability': clipped_p[test_mask],
    'Global logistic calibration': mle.predict_proba(X_base[test_mask])[:, 1],
    'Sport-interaction ridge': ridge_final.predict_proba(X_sport[test_mask])[:, 1],
}
evaluation = pd.DataFrame([
    {'method': name, 'test_market_bin_rows': int(test_mask.sum()),
     'test_markets': model_data.loc[test_mask, 'market_id'].nunique(),
     'test_events': model_data.loc[test_mask, 'event_id'].nunique(),
     'brier': brier_score_loss(model_y[test_mask], pred),
     'log_loss': log_loss(model_y[test_mask], pred, labels=[0, 1])}
    for name, pred in predictions.items()
]).sort_values('log_loss')
display(evaluation.reset_index(drop=True))

## 8 · Interpretation checklist

- Describe the 23-hour results as a **market–bin-weighted retrospective analysis**. Counts across bins can exceed unique markets because a market may visit several bins.
- Compare bin-level gaps, bootstrap intervals and the logistic slope; avoid calling a sport more accurate solely because its Brier score is lower.
- Report how many markets, event links and usable prices were excluded, and check how much the top sports and volume groups differ in probability distribution.
- Check event IDs, sport-label errors, potentially truncated trade histories, and whether event outcomes were already known before a market's recorded `endDate`.
- The event bootstrap assumes clusters are independent; related events sharing teams, tournaments or dates can remain dependent. The chronological evaluation prevents event overlap but does not convert trajectory-based observations into fixed-horizon forecasts.
- Treat final volume associations as descriptive. Historical liquidity and spread are not available from the final metadata snapshot.
- Sensitivity analyses: 24 versus 23 hours; one pre-specified fixed pre-event horizon where reliable event start times exist; alternative bin widths and logit clipping; and exclusion of potentially truncated histories.

### Reproducibility

The methods use the project's `plot_event_sport_calibration.py` and `Polymarket_Closed_Sports_Combined_Pipeline.ipynb` for hourly extraction and Pietro's sport rules. Random seed, bin width, bootstrap iterations, event joins, split and source-file fingerprints are specified above. No source files are modified; only the fingerprinted derived cache is written.